<a href="https://colab.research.google.com/github/MiBC03/life_algorithm/blob/main/life_topk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Rodando em: {device}")

df_name = "movielens_100k_u1.base"
sep = "\t"

# df_name = "ratings_Health_and_Personal_Care.csv"
# sep = ","

# df_name = "ratings_Grocery_and_Gourmet_Food.csv"
# sep = ","

df = pd.read_csv(df_name, sep=sep, header=None)
df.columns = ["userId", "itemId", "rating", "timestamp"]

qtd_users = 943
qtd_items = 1650

df_users = df['userId'].value_counts().nlargest(qtd_users).index
df = df[df['userId'].isin(df_users)]

df_items = df['itemId'].value_counts().nlargest(qtd_items).index
df = df[df['itemId'].isin(df_items)]

df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
df = df.sort_values('timestamp').reset_index(drop=True)

item_ids = df["itemId"].unique()
usuarios_ids = df["userId"].unique()
print(f"Total Itens: {len(item_ids)}")
print(f"Total Usuários: {len(usuarios_ids)}")

categorias_possiveis = [1, 2, 3, 4]
probabilidades = [0.6, 0.2, 0.1, 0.1]

item_to_cat = {
    i: np.random.choice(categorias_possiveis, p=probabilidades)
    for i in item_ids
}

df["categoria"] = df["itemId"].map(item_to_cat)

np.random.seed(42)

ttl_por_item = {
    i: np.random.randint(30, 60)
    for i in item_ids
}

publish_time = df.groupby("itemId")["timestamp"].min().to_dict()

def compute_urgency(item_id, current_time):
    pub_time = publish_time[item_id]
    TTL = ttl_por_item[item_id]
    age_days = (current_time - pub_time).days

    if age_days < 0 or age_days > TTL:
        return 0.0

    urgencia_final = age_days / TTL
    return urgencia_final

In [ ]:
class RatingsDataset(Dataset):
    def __init__(self, df, num_items, num_negatives=4):
        self.users = torch.tensor(df['user_idx'].values, dtype=torch.long)
        self.items = torch.tensor(df['item_idx'].values, dtype=torch.long)
        self.ratings = torch.tensor(df['rating_norm'].values, dtype=torch.float32)
        self.num_items = num_items

        self.num_negatives = num_negatives

    def __len__(self):
        return len(self.ratings) * (1 + self.num_negatives)

    def __getitem__(self, idx):
        real_len = len(self.ratings)

        if idx < real_len:

            return self.users[idx], self.items[idx], self.ratings[idx]
        else:

            orig_idx = idx % real_len
            u = self.users[orig_idx]
            i_neg = torch.randint(0, self.num_items, (1,)).item()
            return u, torch.tensor(i_neg, dtype=torch.long), torch.tensor(0.0, dtype=torch.float32)

class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_items, latent_dim=64):
        super().__init__()
        self.user_matrices = nn.Embedding(num_users, latent_dim)
        self.item_matrices = nn.Embedding(num_items, latent_dim)

        self.user_biases = nn.Embedding(num_users, 1)
        self.item_biases = nn.Embedding(num_items, 1)

        nn.init.normal_(self.user_matrices.weight, 0, 0.1)
        nn.init.normal_(self.item_matrices.weight, 0, 0.1)
        nn.init.zeros_(self.user_biases.weight)
        nn.init.zeros_(self.item_biases.weight)

    def forward(self, u, i):
        dot_product = (self.user_matrices(u) * self.item_matrices(i)).sum(1)
        user_b = self.user_biases(u).squeeze()
        item_b = self.item_biases(i).squeeze()
        return dot_product + user_b + item_b

In [ ]:
def precision_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    if len(recommended_k) == 0: return 0.0
    hits = len(set(recommended_k) & set(relevant))
    return hits / k

def recall_at_k(recommended, relevant, k):
    if len(relevant) == 0: return 0.0
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / len(relevant)

def dcg_at_k(recommended, relevant, k):
    dcg = 0.0
    for idx, item in enumerate(recommended[:k]):
        if item in relevant:
            dcg += 1 / np.log2(idx + 2)
    return dcg

def ndcg_at_k(recommended, relevant, k):
    ideal_dcg = 0.0
    ideal_hits = min(len(relevant), k)
    for i in range(ideal_hits):
        ideal_dcg += 1 / np.log2(i + 2)
    if ideal_dcg == 0: return 0.0
    dcg = dcg_at_k(recommended, relevant, k)
    return dcg / ideal_dcg

def average_precision_at_k(recommended, relevant, k):
    ap = 0.0
    hits = 0
    for idx, item in enumerate(recommended[:k]):
        if item in relevant:
            hits += 1
            ap += hits / (idx + 1)
    if len(relevant) == 0: return 0.0
    return ap / min(len(relevant), k)

def utility_loss(original_scores, reranked_scores):
    original_utility = np.sum(original_scores)
    reranked_utility = np.sum(reranked_scores)
    if original_utility == 0: return 0.0
    loss = (original_utility - reranked_utility) / original_utility
    return loss

In [ ]:
def rerank(df_topk, target_exposure, R, idx_users, inv_idx_items,
           alpha_fair=0.4, beta_urgency=0.2, gamma_relevance=0.4, max_swaps=5):

    df_global = df_topk.copy().reset_index(drop=True)
    todas_categorias = np.array(target_exposure.index, dtype=int)
    max_cat = np.max(todas_categorias)
    num_items = len(inv_idx_items)

    idx_items = {m_id: idx for idx, m_id in inv_idx_items.items()}
    item_cats = np.zeros(num_items, dtype=int)
    item_urgencies = np.zeros(num_items, dtype=float)

    for idx in range(num_items):
        m_id = inv_idx_items[idx]
        item_cats[idx] = item_to_cat[m_id]
        item_urgencies[idx] = compute_urgency(m_id, current_start)

    target_arr = np.zeros(max_cat + 1, dtype=float)
    for cat in todas_categorias:
        target_arr[cat] = target_exposure.get(cat, 0.0)

    current_exp_arr = np.zeros(max_cat + 1, dtype=float)
    for cat in todas_categorias:
        current_exp_arr[cat] = df_global[df_global['categoria'] == cat]['exposure'].sum()

    user_indices = df_global.groupby("userId").groups

    for user_id, idx_user_data in user_indices.items():
        novo_df_user = df_global.loc[idx_user_data].copy()
        uid = idx_users[user_id]
        scores = R[uid].detach().cpu().numpy()
        score_norm = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)

        swaps = 0
        while swaps < max_swaps:
            cats_sub_mask = current_exp_arr < target_arr
            cats_sub = np.where(cats_sub_mask)[0]

            if len(cats_sub) == 0: break

            threshold = novo_df_user["score"].quantile(0.6)
            cats_over_mask = current_exp_arr > target_arr

            categorias_user = novo_df_user["categoria"].astype(int).values
            pode_sair_mask = (novo_df_user["score"].values <= threshold) & cats_over_mask[categorias_user]

            candidatos_sair = novo_df_user.iloc[pode_sair_mask]
            if candidatos_sair.empty: break

            idx_sair = candidatos_sair.sample(1).index[0]
            item_sair = novo_df_user.loc[idx_sair]

            W = float(item_sair["exposure"])
            cat_out = int(item_sair["categoria"])

            itens_no_topk_ids = novo_df_user["itemId"].values
            itens_no_topk_idx = np.array([idx_items[m_id] for m_id in itens_no_topk_ids])

            valid_mask = np.isin(item_cats, cats_sub)
            valid_mask[itens_no_topk_idx] = False

            valid_indices = np.where(valid_mask)[0]
            if len(valid_indices) == 0: break

            c_cats = item_cats[valid_indices]
            c_urgencies = item_urgencies[valid_indices]
            c_scores_norm = score_norm[valid_indices]

            current_out = current_exp_arr[cat_out]
            target_out = target_arr[cat_out]
            current_in_arr = current_exp_arr[c_cats]
            target_in_arr = target_arr[c_cats]

            loss_before = abs(current_out - target_out) + np.abs(current_in_arr - target_in_arr)
            loss_after = abs((current_out - W) - target_out) + np.abs((current_in_arr + W) - target_in_arr)

            fairness_gain = loss_before - loss_after

            fg_min, fg_max = np.min(fairness_gain), np.max(fairness_gain)
            if fg_max > fg_min:
                fairness_gain_norm = (fairness_gain - fg_min) / (fg_max - fg_min)
            else:
                fairness_gain_norm = np.zeros_like(fairness_gain)

            prioridades = (alpha_fair * fairness_gain_norm) + (beta_urgency * c_urgencies) + (gamma_relevance * c_scores_norm)

            best_local_idx = np.argmax(prioridades)
            best_idx = valid_indices[best_local_idx]

            best_m_id = inv_idx_items[best_idx]
            best_cat = c_cats[best_local_idx]
            best_urgency = c_urgencies[best_local_idx]

            novo_df_user.loc[idx_sair, "itemId"] = best_m_id
            novo_df_user.loc[idx_sair, "score"] = float(scores[best_idx])
            novo_df_user.loc[idx_sair, "categoria"] = best_cat
            novo_df_user.loc[idx_sair, "urgency"] = best_urgency

            current_exp_arr[cat_out] -= W
            current_exp_arr[best_cat] += W
            swaps += 1

        df_global.loc[idx_user_data] = novo_df_user

    return df_global

In [ ]:
# WINDOW_DAYS, WARMUP_DAYS = 30, 180
WINDOW_DAYS, WARMUP_DAYS = 7, 30
window_size = pd.Timedelta(days=WINDOW_DAYS)
warmup_size = pd.Timedelta(days=WARMUP_DAYS)
MIN_USUARIOS_ATIVOS = 100

start_time = df['timestamp'].min()
end_time   = df['timestamp'].max()
current_start = start_time + warmup_size

all_users, all_items = df['userId'].unique(), df['itemId'].unique()
idx_users = {u: i for i, u in enumerate(all_users)}
idx_items = {m: i for i, m in enumerate(all_items)}
inv_idx_items = {i: m for m, i in idx_items.items()}

model = MatrixFactorization(len(all_users), len(all_items)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-5)
loss_fn = nn.BCEWithLogitsLoss()

results_fairness = []
results_quality = []

while current_start + window_size <= end_time:

    current_end = current_start + window_size

    while True:
        df_window = df[(df['timestamp'] >= current_start) & (df['timestamp'] < current_end)]
        usuarios_na_janela = df_window['userId'].nunique()

        if usuarios_na_janela >= MIN_USUARIOS_ATIVOS or current_end > end_time:
            break

        current_end += window_size

    if usuarios_na_janela == 0:
        print(f"Janela: {current_start.date()} → {current_end.date()} | VAZIA - Encerrando.")
        break

    print(f"Janela: {current_start.date()} → {current_end.date()} | Usuários ativos: {usuarios_na_janela}")

    df_train = df[df['timestamp'] < current_start].copy()
    if df_train.empty:
        current_start += window_size
        continue

    df_train['user_idx'] = df_train['userId'].map(idx_users)
    df_train['item_idx'] = df_train['itemId'].map(idx_items)

    min_r, max_r = df_train['rating'].min(), df_train['rating'].max()
    df_train['rating_norm'] = (df_train['rating'] - min_r) / (max_r - min_r + 1e-9)

    loader = DataLoader(RatingsDataset(df_train, len(all_items), num_negatives=4), batch_size=512, shuffle=True)

    model.train()
    for _ in range(10):
        for u, i, r in loader:
            u, i, r = u.to(device), i.to(device), r.to(device)

            optimizer.zero_grad()
            loss = loss_fn(model(u, i), r)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        U = model.user_matrices.weight
        I = model.item_matrices.weight
        B_u = model.user_biases.weight
        B_i = model.item_biases.weight

        R_original = torch.matmul(U, I.t()) + B_u + B_i.t()

    rows = []
    k = 10

    # Construindo rankings
    for u in df_window['userId'].unique():
        uid = idx_users[u]

        scores_adj = R_original[uid].clone()

        itens_vistos = df_train[df_train['userId'] == u]['item_idx'].values
        scores_adj[itens_vistos] = -999999

        topk = torch.topk(scores_adj, k)

        rank_pos = 1
        for idx_item in topk.indices.cpu().numpy():
            m_id = inv_idx_items[idx_item]

            rows.append({
                'userId': u,
                'itemId': m_id,
                'score': R_original[uid][idx_item].item(),
                'rank': rank_pos,
                'categoria': item_to_cat[m_id],
                'urgency': compute_urgency(m_id, current_start)
            })

            rank_pos += 1
            if rank_pos > k: break

    df_topk = pd.DataFrame(rows)

    if not df_topk.empty:

        df_topk['exposure'] = 1 / np.log2(df_topk['rank'] + 1)

        # Calculando Target Exposure
        num_usuarios_ativos = df_topk['userId'].nunique()
        pesos_k = 1 / np.log2(np.arange(1, k + 1) + 1)
        E = pesos_k.sum() * num_usuarios_ativos

        itens_ativos_na_janela = df_window['itemId'].unique()
        indices_ativos = [idx_items[m_id] for m_id in itens_ativos_na_janela]

        scores_globais = R_original.sum(dim=0).cpu().numpy()
        scores_ativos = scores_globais[indices_ativos]

        df_scores = pd.DataFrame({
            'itemId': itens_ativos_na_janela,
            'score_total': scores_ativos
        })
        df_scores['categoria'] = df_scores['itemId'].map(item_to_cat)
        qi = df_scores.groupby('categoria')['score_total'].sum()
        Q = qi.sum()

        target_exp = E * (qi / Q)
        target_exp = target_exp.reindex(categorias_possiveis, fill_value=0.0)

        df_topk_reranked = rerank(df_topk, target_exp, R_original, idx_users, inv_idx_items)

        usuarios_avaliacao = df_window['userId'].unique()

        metricas_original = {'precision': [], 'recall': [], 'ndcg': [], 'map': []}
        metricas_rerank = {'precision': [], 'recall': [], 'ndcg': [], 'map': []}
        utility_losses = []

        for user_id in usuarios_avaliacao:

            itens_relevantes = set(
                df_window[(df_window['userId'] == user_id) & (df_window['rating'] >= 4)]['itemId'].values
            )

            if len(itens_relevantes) == 0: continue

            ranking_original = df_topk[df_topk['userId'] == user_id].sort_values('rank')
            itens_original = ranking_original['itemId'].tolist()

            ranking_rerank = df_topk_reranked[df_topk_reranked['userId'] == user_id].sort_values('rank')
            itens_rerank = ranking_rerank['itemId'].tolist()

            metricas_original['precision'].append(precision_at_k(itens_original, itens_relevantes, k))
            metricas_original['recall'].append(recall_at_k(itens_original, itens_relevantes, k))
            metricas_original['ndcg'].append(ndcg_at_k(itens_original, itens_relevantes, k))
            metricas_original['map'].append(average_precision_at_k(itens_original, itens_relevantes, k))

            metricas_rerank['precision'].append(precision_at_k(itens_rerank, itens_relevantes, k))
            metricas_rerank['recall'].append(recall_at_k(itens_rerank, itens_relevantes, k))
            metricas_rerank['ndcg'].append(ndcg_at_k(itens_rerank, itens_relevantes, k))
            metricas_rerank['map'].append(average_precision_at_k(itens_rerank, itens_relevantes, k))

            original_scores = ranking_original['score'].values
            rerank_scores = ranking_rerank['score'].values
            utility_losses.append(utility_loss(original_scores, rerank_scores))

        if len(utility_losses) > 0:
            results_quality.append({
                'start': current_start,
                'precision_original': np.mean(metricas_original['precision']),
                'precision_rerank': np.mean(metricas_rerank['precision']),
                'recall_original': np.mean(metricas_original['recall']),
                'recall_rerank': np.mean(metricas_rerank['recall']),
                'ndcg_original': np.mean(metricas_original['ndcg']),
                'ndcg_rerank': np.mean(metricas_rerank['ndcg']),
                'map_original': np.mean(metricas_original['map']),
                'map_rerank': np.mean(metricas_rerank['map']),
                'utility_loss': np.mean(utility_losses)
            })

        exposure_before = df_topk.groupby('categoria')['exposure'].sum()
        exposure_before /= exposure_before.sum()

        exposure_after = df_topk_reranked.groupby('categoria')['exposure'].sum()
        exposure_after /= exposure_after.sum()

        target_exp_norm = target_exp / target_exp.sum()

        results_fairness.append({
            'start': current_start,
            'target': target_exp_norm.to_dict(),
            'before': exposure_before.to_dict(),
            'after': exposure_after.to_dict(),
        })

    current_start = current_end

In [ ]:
if results_fairness:
    total_target = {}
    total_before = {}
    total_after = {}
    n_janelas = len(results_fairness)

    for res in results_fairness:
        for cat, val in res['target'].items():
            total_target[cat] = total_target.get(cat, 0) + val
        for cat, val in res['before'].items():
            total_before[cat] = total_before.get(cat, 0) + val
        for cat, val in res['after'].items():
            total_after[cat] = total_after.get(cat, 0) + val

    categorias = sorted(total_target.keys())
    global_metrics = []

    for cat in categorias:
        global_metrics.append({
            'Categoria': cat,
            'Average Target (%)': round((total_target.get(cat, 0) / n_janelas) * 100, 2),
            'Average Top-K (%)': round((total_before.get(cat, 0) / n_janelas) * 100, 2),
            'Average Rerank (%)': round((total_after.get(cat, 0) / n_janelas) * 100, 2)
        })

    df_global = pd.DataFrame(global_metrics)

    print(f"Exposição - Média ({n_janelas} janelas)")
    print(df_global.to_string(index=False))



In [ ]:
if len(results_quality) > 0:
    print("Métricas do usuário")

    precision_original = np.mean([r['precision_original'] for r in results_quality])
    precision_rerank = np.mean([r['precision_rerank'] for r in results_quality])

    recall_original = np.mean([r['recall_original'] for r in results_quality])
    recall_rerank = np.mean([r['recall_rerank'] for r in results_quality])

    ndcg_original = np.mean([r['ndcg_original'] for r in results_quality])
    ndcg_rerank = np.mean([r['ndcg_rerank'] for r in results_quality])

    map_original = np.mean([r['map_original'] for r in results_quality])
    map_rerank = np.mean([r['map_rerank'] for r in results_quality])

    utility_loss_mean = np.mean([r['utility_loss'] for r in results_quality])

    print(f"\nPrecision@10")
    print(f"Original : {precision_original:.4f}")
    print(f"Rerank   : {precision_rerank:.4f}")

    print(f"\nRecall@10")
    print(f"Original : {recall_original:.4f}")
    print(f"Rerank   : {recall_rerank:.4f}")

    print(f"\nNDCG@10")
    print(f"Original : {ndcg_original:.4f}")
    print(f"Rerank   : {ndcg_rerank:.4f}")

    print(f"\nMAP@10")
    print(f"Original : {map_original:.4f}")
    print(f"Rerank   : {map_rerank:.4f}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

if results_fairness and len(results_quality) > 0:
    n_janelas = len(results_fairness)

    total_target, total_before, total_after = {}, {}, {}
    for res in results_fairness:
        for cat, val in res['target'].items(): total_target[cat] = total_target.get(cat, 0) + val
        for cat, val in res['before'].items(): total_before[cat] = total_before.get(cat, 0) + val
        for cat, val in res['after'].items(): total_after[cat] = total_after.get(cat, 0) + val

    categorias = sorted(total_target.keys())
    exp_linhas = []

    for cat in categorias:
        tgt = (total_target.get(cat, 0) / n_janelas) * 100
        bef = (total_before.get(cat, 0) / n_janelas) * 100
        aft = (total_after.get(cat, 0) / n_janelas) * 100
        exp_linhas.append({
            'Category': f'Cat {cat}',
            'Orig. Exp.': f'{bef:.2f}%',
            'Rerank Exp.': f'{aft:.2f}%',
            'Target Exp.': f'{tgt:.2f}%',
            'Exp. Change': f'{(aft - bef):+0.2f} p.p.'
        })

    qual_linhas = []
    metricas_map = {
        'Precision@10': ('precision_original', 'precision_rerank'),
        'Recall@10': ('recall_original', 'recall_rerank'),
        'NDCG@10': ('ndcg_original', 'ndcg_rerank'),
        'MAP@10': ('map_original', 'map_rerank')
    }

    for nome, (key_orig, key_rerank) in metricas_map.items():
        val_orig = np.mean([r[key_orig] for r in results_quality])
        val_rerank = np.mean([r[key_rerank] for r in results_quality])

        queda_pct = ((val_rerank - val_orig) / val_orig) * 100 if val_orig != 0 else 0
        qual_linhas.append({
            'Quality Metric': nome,
            'Drop Proportion': f'{queda_pct:+.2f}%'
        })

    max_len = max(len(exp_linhas), len(qual_linhas))

    while len(exp_linhas) < max_len:
        exp_linhas.append({'Category': '', 'Orig. Exp.': '', 'Rerank Exp.': '', 'Target Exp.': '', 'Exp. Change': ''})
    while len(qual_linhas) < max_len:
        qual_linhas.append({'Quality Metric': '', 'Drop Proportion': ''})

    linhas_combinadas = []
    for i in range(max_len):
        row = {}
        row.update(exp_linhas[i])
        row['   '] = ''
        row.update(qual_linhas[i])
        linhas_combinadas.append(row)

    df_resumo = pd.DataFrame(linhas_combinadas)

    print(f"=== Exposure and Quality Side-by-Side ({n_janelas} windows) ===\n")
    print(df_resumo.to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, 2))
    ax.axis('tight')
    ax.axis('off')

    tabela = ax.table(cellText=df_resumo.values,
                      colLabels=df_resumo.columns,
                      loc='center',
                      cellLoc='center')

    tabela.auto_set_font_size(False)
    tabela.set_fontsize(10)
    tabela.scale(1.6, 1.6)

    for i, col_name in enumerate(df_resumo.columns):
        if col_name == '   ':
            tabela[(0, i)].set_edgecolor('white')
            tabela[(0, i)].set_facecolor('white')
            tabela[(0, i)].set_width(0.02)

            for j in range(1, len(df_resumo) + 1):
                tabela[(j, i)].set_edgecolor('white')
                tabela[(j, i)].set_facecolor('white')
                tabela[(j, i)].set_width(0.02)
        else:
            tabela[(0, i)].set_facecolor("#e8e8e8")
            tabela[(0, i)].set_text_props(weight='bold')

    plt.savefig('tabela_resultados.png', dpi=300, bbox_inches='tight')
    plt.show()